# 03 — Supervised Model Training

Train baseline (LogisticRegression) and improved (LightGBM) models for delinquency, default, prepayment, and next-state prediction using a time-aware split.

In [ ]:
import sys, warnings, json
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.evaluation.time_split import time_aware_split, get_split_masks
from src.features.engineer import LeakageSafeFeatureEngineer

train_df = pd.read_csv('../data/synthetic/loan_monthly_performance_train.csv')
for col in ['reporting_month','origination_month']:
    if col in train_df.columns:
        train_df[col] = pd.to_datetime(train_df[col]).dt.to_period('M')
servicer_df = pd.read_csv('../data/synthetic/servicer_updates.csv')
from src.pipeline.loader import reconcile_servicer_updates
train_df, _ = reconcile_servicer_updates(train_df, servicer_df)

fe = LeakageSafeFeatureEngineer()
X_full = fe.fit_transform(train_df, is_train=True)
target_cols = ['next_3m_delinquency_flag','next_6m_delinquency_flag',
               'next_12m_default_flag','next_12m_prepayment_flag','next_state']
y_full = train_df[target_cols]

train_mask, val_mask, test_mask = get_split_masks(train_df)
X_tr, X_val, X_te = X_full[train_mask], X_full[val_mask], X_full[test_mask]
y_tr, y_val, y_te = y_full[train_mask], y_full[val_mask], y_full[test_mask]
print(f'Train={len(X_tr):,}  Val={len(X_val):,}  Test={len(X_te):,}')


## Train models

In [ ]:
from src.modeling.train_supervised import train_classification_models
from src.utils.config import get_settings
config = get_settings('../config/settings.yaml')
clf_trainer = train_classification_models(X_tr, y_tr, X_val, y_val, X_te, y_te, config)
print('Training complete.')


## Model Comparison

In [ ]:
results = json.load(open('../reports/modeling/detailed_results.json'))
rows = []
for tgt, models in results.items():
    for m in models:
        vm = m.get('val_metrics', {})
        rows.append({'target': tgt, 'model': m['model_name'],
                     'val_ROC-AUC': round(vm.get('roc_auc', float('nan')), 4),
                     'val_PR-AUC': round(vm.get('pr_auc', float('nan')), 4),
                     'val_F1': round(vm.get('f1', float('nan')), 4),
                     'val_Brier': round(vm.get('brier_score', float('nan')), 4)})
comp_df = pd.DataFrame(rows)
print(comp_df.to_string(index=False))


## Baseline vs Improved Lift

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pivot = comp_df.pivot(index='target', columns='model', values='val_ROC-AUC')
pivot.plot(kind='bar', ax=axes[0], color=['#E87070','#4C9BE8'])
axes[0].set_title('Validation ROC-AUC: Baseline vs Improved')
axes[0].set_ylim(0.5, 1.0)
axes[0].tick_params(axis='x', rotation=20)
axes[0].legend()

pivot2 = comp_df.pivot(index='target', columns='model', values='val_PR-AUC')
pivot2.plot(kind='bar', ax=axes[1], color=['#E87070','#4C9BE8'])
axes[1].set_title('Validation PR-AUC: Baseline vs Improved')
axes[1].set_ylim(0, 1.0)
axes[1].tick_params(axis='x', rotation=20)
axes[1].legend()
plt.tight_layout()
plt.savefig('../reports/model_comparison_chart.png', dpi=150)
plt.show()
